In [ ]:
import os
from pathlib import Path
import numpy as np
import cv2
import albumentations as A
from ultralytics import YOLO
import shutil

In [ ]:
# paths and config
BASE_PATH = Path("../data")
EXP_PATH = BASE_PATH / "EXPERIMENT1"
MODELS_PATH = Path("../models")
BASE_MODELS_PATH = MODELS_PATH / "base_models"
DATASET_ROOT = BASE_PATH / "EXPERIMENT1/domain_adaptation/train" # public dataset / must be changed for second experiment CHANGEEEEEEEEEE
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
CLASS_NAMES = ['abiotic_stress', 'bacterial_spot', 'chewing_insect', 'healthy', 'mechanical_stress', 'mite_presence']
NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
TRAIN_YOLO_DIR = Path("/content/drive/MyDrive/OsirisProject/development_files/EXPERIMENT1/domain_adaptation/dataset_YOLO/train")
os.makedirs(TRAIN_YOLO_DIR, exist_ok=True)


CUSTOM_AUGMENTATION= A.Compose([

    A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5), 
    A.RandomResizedCrop(
      size=(IMG_SIZE[0], IMG_SIZE[1]),
      scale=(0.9, 1.0),
      ratio=(0.9, 1.1),
      interpolation=cv2.INTER_LINEAR,
      p=0.3
  ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),

    A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.3),

    A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=5, val_shift_limit=4, p=0.2),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=0,
        border_mode=cv2.BORDER_CONSTANT,
        value=0, 
        p=0.3
    ),

    A.Resize(width=IMG_SIZE[1], height=IMG_SIZE[0]) 
], p=1.0)

In [ ]:
def generate_and_save_augmented_data_oversample(input_folder, output_folder, augmentation_pipeline, min_per_class=400):
    """
    Generates and saves augmented images from an input folder,
    applying oversampling to minority classes to reach a minimum
    of 'min_per_class' images per class.

    :param input_folder: Folder with class subdirectories (e.g.: train_dir).
    :param output_folder: Folder where augmented data will be saved (e.g.: train_aug).
    :param augmentation_pipeline: Albumentations pipeline.
    :param min_per_class: Minimum target number of images per class.
    """
    input_path = Path(input_folder)
    output_path = Path(output_folder)

    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    classes = sorted([d.name for d in input_path.iterdir() if d.is_dir()])

    total_original_images = 0
    total_augmented_saved = 0
    class_stats = {}

    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]

    print(f"Classes found: {classes}")

    for cls in classes:
        class_input_path = input_path / cls
        class_output_path = output_path / cls
        class_output_path.mkdir(parents=True, exist_ok=True)

        image_paths = []
        for e in exts:
            image_paths.extend(list(class_input_path.glob(e)))

        n_original = len(image_paths)
        total_original_images += n_original
        n_saved_current_class = 0

        if n_original == 0:
            print(f"[WARNING] Class '{cls}' has no images. Skipping augmentation.")
            class_stats[cls] = {'original': 0, 'total_saved': 0, 'augment_factor': 0}
            continue


        if n_original < min_per_class:
            # calculate how many augmented copies are needed per original image
            n_augment_needed = max(0, int(np.ceil(min_per_class / n_original)) - 1)
        else:
            n_augment_needed = 0

        class_stats[cls] = {'original': n_original, 'total_saved': 0, 'augment_factor': n_augment_needed}

        for img_path in image_paths:
            try:
                image = cv2.imread(str(img_path))
                if image is None:
                    print(f"[ERROR] Could not read {img_path}. Skipping.")
                    continue

                cv2.imwrite(str(class_output_path / img_path.name), image)
                n_saved_current_class += 1
                total_augmented_saved += 1

                for i in range(n_augment_needed):
                    augmented = augmentation_pipeline(image=image)
                    augmented_image = augmented['image']

                    original_name = img_path.stem
                    extension = img_path.suffix
                    new_filename = f"{original_name}_aug_{i+1}{extension}"

                    # save the augmented image
                    cv2.imwrite(str(class_output_path / new_filename), augmented_image)
                    n_saved_current_class += 1
                    total_augmented_saved += 1

            except Exception as e:
                print(f"[ERROR] Processing {img_path} -> {e}")

        class_stats[cls]['total_saved'] = n_saved_current_class

    print(f"\n--- Generation Summary (Target: {min_per_class} per class) ---")
    for cls, stats in class_stats.items():
        factor_str = f"({stats['augment_factor']}x)" if stats['augment_factor'] > 0 else "(no oversample)"
        print(f"| {cls:<15}: Original={stats['original']:<4}, Total Saved={stats['total_saved']:<4} {factor_str}")

    print(f"--- Global ---")
    print(f"Total original images found: {total_original_images}")
    print(f"Total images saved (Original + Augmented): {total_augmented_saved}")
    print(f"Augmented data ready in: {output_folder}")

    return class_stats

# call
stats = generate_and_save_augmented_data_oversample(
    input_folder=DATASET_ROOT,
    output_folder=TRAIN_YOLO_DIR,
    augmentation_pipeline=CUSTOM_AUGMENTATION,
    min_per_class=400 # target: 400 images per class
)

# Training YOLOv11: nano (n), medium (m) and large (l) versions.

In [ ]:
DATASET_ROOT = EXP_PATH / "domain_adaptation" / "dataset_YOLO"

In [ ]:

MODELS_TO_TRAIN = {
    "n": {"batch": 32},
    "m": {"batch": 32},
    "l": {"batch": 16}
}

BASE_TRAIN_PARAMS = {
    'data': DATASET_ROOT,
    'epochs': 30,
    'imgsz': 224,
    'device': 0,
    'project': MODELS_PATH,
    'workers': 4,
    'copy_paste': 0.0, 'mosaic': 0.0, 'mixup': 0.0, 'fliplr': 0.0,
    'flipud': 0.0, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
}

for suffix, config in MODELS_TO_TRAIN.items():
    model_name = f"yolo11{suffix}-cls"
    print(f"\n{'='*50}\nStarting training: {model_name}\n{'='*50}")
    
    model = YOLO(f"{model_name}.pt")
    
    current_params = BASE_TRAIN_PARAMS.copy()
    current_params.update(config)
    current_params['name'] = f"YOLO11{suffix}_cls"

    model.train(**current_params)
    
    print(f"\n--- Training completed: {model_name} ---")


(PASAR PARTE DE EVALUACIÓN A RESULTS)